# 02 - Cargar datastore y mosaic dataset

Segunda etapa del flujo Geosupport. Este notebook usa el `04_ready_for_datastore.csv` generado en la etapa 1 para copiar las imagenes al datastore, agregarlas al mosaic dataset, construir footprints y actualizar campos criticos.

Ejecutar primero con `DRY_RUN = True`. Cuando la revision sea correcta, cambiar a `DRY_RUN = False`.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import process_mosaic_load_row

print('Modulo carga:', mosaic_loader.__file__)

## Parametros

La entrada normal es el CSV `04_ready_for_datastore.csv` de la etapa 1. Esta etapa genera sus propios archivos de resultado en una carpeta fija, reemplazando resultados anteriores.

In [ ]:
READY_FOR_DATASTORE_CSV = Path.cwd() / 'outputs' / 'etapa_01_preparar_paths_datastore' / '04_ready_for_datastore.csv'

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"
IMAGE_SERVICE_NAME = 'CL_MLP_PAO_IF_Ortho_Geosupport'

PROJECT_VALUE = 'PAO'
SENSOR_VALUE = 'DJI Mavic Enterprise'
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15
MINPS_VALUE = 0

# Seguridad operacional.
DRY_RUN = True
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Usar None para procesar todo. Para prueba controlada usar, por ejemplo, 1 o 5.
LIMIT_ROWS = None

run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path.cwd() / 'outputs' / 'etapa_02_carga_datastore_mosaico'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('CSV entrada:', READY_FOR_DATASTORE_CSV)
print('Mosaic dataset:', PATH_MOSAIC_DATASET)
print('Salida:', OUTPUT_DIR)
print('DRY_RUN:', DRY_RUN)
print('OVERWRITE_COPY:', OVERWRITE_COPY)
print('SKIP_EXISTING_MOSAIC_NAME:', SKIP_EXISTING_MOSAIC_NAME)
print('LIMIT_ROWS:', LIMIT_ROWS)

## 1. Leer manifiesto listo de etapa 1

Se procesan solo registros con `ready_for_datastore = True`. El CSV de entrada debe tener origen (`path`) y destino (`destination_path`).

In [ ]:
ready_df = pd.read_csv(READY_FOR_DATASTORE_CSV)

required_columns = ['path', 'destination_path', 'expected_name', 'expected_file_name', 'expected_date_token', 'spatial_sector_raw']
missing_columns = [column for column in required_columns if column not in ready_df.columns]
if missing_columns:
    raise ValueError(f'Faltan columnas requeridas en {READY_FOR_DATASTORE_CSV}: {missing_columns}')

if 'ready_for_datastore' in ready_df.columns:
    ready_df = ready_df[ready_df['ready_for_datastore'].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()

if LIMIT_ROWS is not None:
    ready_df = ready_df.head(int(LIMIT_ROWS)).copy()

print(f'Registros a procesar: {len(ready_df)}')
display(ready_df[['file_name', 'expected_file_name', 'destination_path', 'spatial_sector_raw']].head(20))

## 2. Preparar campos criticos del mosaic dataset

Los campos se derivan del manifest de etapa 1 y de valores fijos del proyecto. La URL usa la ruta relativa dentro del datastore.

In [ ]:
def date_token_to_iso(date_token):
    if not date_token or pd.isna(date_token):
        return None
    parts = str(date_token).split('_')
    if len(parts) != 3:
        return None
    year, month, day = parts
    return f'20{year}-{month}-{day}'


def build_imageserver_url(row):
    if not row.get('destination_folder') or not row.get('destination_date_folder') or not row.get('expected_file_name'):
        return None
    relative_file_id = f".\\{row['destination_folder']}\\{row['destination_date_folder']}\\{row['expected_file_name']}"
    return f'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/{IMAGE_SERVICE_NAME}/ImageServer/file?id={relative_file_id}&rasterId='


load_df = ready_df.copy()
load_df['Name'] = load_df['expected_name']
load_df['Raster'] = load_df['expected_file_name']
load_df['Path_Destino'] = load_df['destination_path']
load_df['Sector'] = load_df['spatial_sector_raw']
load_df['Fecha_Adqui'] = load_df['expected_date_token'].map(date_token_to_iso)
load_df['URL'] = load_df.apply(build_imageserver_url, axis=1)
load_df['Proyecto'] = PROJECT_VALUE
load_df['Sensor'] = SENSOR_VALUE
load_df['Fecha_Publ'] = datetime.now().strftime('%Y-%m-%d')
load_df['MaxPS'] = MAXPS_VALUE
load_df['LowPS'] = LOWPS_VALUE
load_df['MinPS'] = MINPS_VALUE
load_df['ProductName'] = 'OBJECTID del registro en el mosaic dataset'

critical_columns = ['Name', 'Raster', 'Path_Destino', 'Sector', 'Fecha_Adqui', 'URL', 'Proyecto', 'Sensor', 'Fecha_Publ', 'MaxPS', 'LowPS', 'MinPS', 'ProductName']
missing_critical = load_df[critical_columns].isna().sum().reset_index(name='null_count').rename(columns={'index': 'field'})

display(load_df[['file_name'] + critical_columns].head(20))
display(missing_critical)

## 3. Ejecutar copia, carga al mosaico, footprint y atributos

Secuencia por cada imagen: copiar al datastore, agregar raster al mosaic dataset, construir footprint y actualizar atributos. Si `DRY_RUN = True`, no se escribe en el datastore ni en el mosaico.

In [ ]:
results = []

for index, row in load_df.iterrows():
    print(f"{index + 1}/{len(load_df)} - {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        minps_value=MINPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df.head(30))

for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        display(results_df[column].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': column}))

## 4. Exportar resultados de la etapa

Los resultados se escriben en una carpeta fija de la etapa y reemplazan los CSV anteriores.

In [ ]:
summary_rows = [
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'ready_for_datastore_csv', 'value': str(READY_FOR_DATASTORE_CSV)},
    {'metric': 'mosaic_dataset', 'value': PATH_MOSAIC_DATASET},
    {'metric': 'dry_run', 'value': DRY_RUN},
    {'metric': 'overwrite_copy', 'value': OVERWRITE_COPY},
    {'metric': 'skip_existing_mosaic_name', 'value': SKIP_EXISTING_MOSAIC_NAME},
    {'metric': 'maxps_value', 'value': MAXPS_VALUE},
    {'metric': 'lowps_value', 'value': LOWPS_VALUE},
    {'metric': 'minps_value', 'value': MINPS_VALUE},
    {'metric': 'product_name_value', 'value': 'OBJECTID del registro en el mosaic dataset'},
    {'metric': 'rows_to_process', 'value': len(load_df)},
]

for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({'metric': f'{column}_{status}', 'value': int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / '00_summary.csv'
load_input_csv = OUTPUT_DIR / '01_load_input_with_attributes.csv'
load_results_csv = OUTPUT_DIR / '02_load_results.csv'
errors_csv = OUTPUT_DIR / '03_errors_review.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
load_df.to_csv(load_input_csv, index=False, encoding='utf-8-sig')
results_df.to_csv(load_results_csv, index=False, encoding='utf-8-sig')

error_mask = pd.Series(False, index=results_df.index)
for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        error_mask = error_mask | results_df[column].astype(str).str.contains('error|failed|missing|not_found', case=False, na=False)
results_df[error_mask].to_csv(errors_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)
print('Input enriquecido:', load_input_csv)
print('Resultados carga:', load_results_csv)
print('Revision errores:', errors_csv)